# 📝 ksnctf 36「Are you ESPer?」ローカル環境検証記録

## 1. 問題の概要と本質
本問題は、1〜9の数字から正解を当てるゲーム（全20ステージ）をノーミスでクリアする問題である。
バイナリ（`esp`）を静的解析した結果、ステージ開始時に `srandom(time(NULL));` で現在時刻（Unixタイムスタンプ）をシード値にした乱数が生成されていることが判明した。

$$var1 = rand() \pmod{10}$$

理論上、サーバーと「完全に同じ時刻（シード値）」を共有し、同じ乱数生成アルゴリズムを動かせば、超能力（ESP）を使わずとも100%の精度で全問正解できる。

---

## 2. 構築した検証環境のアーキテクチャ
手元のWindows環境から、OSの壁や環境のノイズを排除して確実に再現・検証を行うため、以下の構成を構築した。

* **Jupyter環境**: 実験の起点およびコード・Dockerfileの管理
* **Dockerコンテナ（Ubuntu 22.04）**: 32bit互換ライブラリ（`libc6-i386`）を導入し、本物の `esp` バイナリを `socat`（ポート `10777`）でネットワーク中継して待ち受けるサーバー環境
* **WSL2環境（Ubuntu）**: 本物のLinux C標準ライブラリ（`libc`）の `rand()` を直接呼び出してサーバーと同期する、Rust言語を用いた超高速ソルバーの実行環境

---

## 3. 直面した課題とトラブルシューティングの軌跡

### 🛑 課題1: Windows（Jupyter）上での乱数再現バグ
最初、Jupyter（Windows）上のRustコードでLinuxの `rand()`（glibcの線形合同法および状態遷移アルゴリズム）を自作構造体 `GlibcRand` で完全再現しようとした。
しかし、Rustの演算子優先順位（`% 10` と `.abs()` の関係）や、符号付き32bit整数のキャストの挙動により、**送信する値がすべて `0` や `2` に固定されてしまうバグ**が発生した。

* **対策**: 計算順序を `(rng.next().abs() % 10)` に修正。しかし、根本的な解決とコードの簡素化のため、**WSL2（Linux）上のRustからC言語の本物の `libc` ライブラリを直接インポートする方式**へ転換した。

### 🛑 課題2: サーバーとの時間差（時差）の再現エラー
本家のksnctfサーバーは、NTP同期がされておらずOSの時計が数十秒〜数十分ズレている（クロックドリフト）。これを再現するため、`Dockerfile` 内の `CMD` で `date -s` 命令を使い強制的にコンテナ時間を歪めようとした。
しかし、Windows側のJupyterエディタでエスケープ文字（`\`）が「円マーク（`￥`）」に文字化けし、コンテナ起動時に `date` コマンドがエラーを起こして `Sun May 17...` というエラーログを出力。これが `socat` の通信路（標準出力）にノイズとして混ざり、Rust側がフリーズする原因となった。

* **対策**: 外部コマンド（`date`）に頼るのを完全にやめ、**コンテナの環境変数 `TZ=America/New_York` を指定して「ニューヨーク時間（13時間＝46800秒の巨大な時差）」を意図的に作り出す、タイムゾーンハックを採用した。**

### 🛑 課題3: FLAG提供時の音信不通（空白）現象
Level 20の最終問題をクリアしても、画面にフラグが表示されずプログラムが終了した。
`strings esp` コマンドによるバイナリ内の文字列抽出、および `objdump -d -M intel esp` によるアセンブリの紐解きにより、原因を特定した。

```assembly
80483f9:  mov    DWORD PTR [esp],0x80bb210     ; アドレス 0x80bb210 = "/home/q36/flag.txt"
8048400:  call   8055930 <_IO_new_fopen>       ; fopenを実行